# Simulation study II — multi-modal posteriors and ESS (paper §5.3)

Reproduces the LIT-MH-1 column of the paper's Table 2: sampling efficiency on posterior distributions with varying degrees of multi-modality, controlled by $\sigma_eta$.

**Data-generating model** (§5.3): rows of $X$ i.i.d. from $\mathrm{MN}(0, \Sigma_{d,p})$, block-diagonal with $d 	imes d$ blocks $(\Sigma_d)_{jk} = e^{-|j-k|/3}$; $n = 1000$, $p = 5000$, $d = 20$; $\gamma^*$ uniform on $\{\gamma: |\gamma| = 100\}$ and $eta^*_{\gamma^*} \sim \mathrm{MN}(0, \sigma_eta^2 I_{100})$; $z \sim \mathrm{MN}(0, I_n)$. Prior hyperparameters $\kappa_0 = 1$, $\kappa_1 = 1/2$. The chain runs 2,000 iterations and is initialized at a forward–backward stepwise model (greedy single-covariate add/delete while the posterior increases; a simplified version of An et al. 2008). The posterior is most multi-modal around $\sigma_eta = 0.2$.

**Statistics** (Table 2 style): wall time; mean model size; number of unique single-flip local modes sampled ($\pi_n(\gamma) > \pi_n(\gamma')$ for every $\gamma'$ differing by one covariate); acceptance rate (approximated by the fraction of iterations whose model changed); and effective sample sizes of the one-dimensional summaries
$$T_1^{(k)} = \lVert \hateta(\gamma^{(k)}) - eta^* 
Vert_2^2, \qquad T_2^{(k)} = \lVert X eta^{(k)} 
Vert_2^2,$$
where $\hateta(\gamma) = \mathbb{E}[eta \mid \gamma, y]$ and $eta^{(k)}$ is drawn from the conditional posterior — $T_1$ reflects the sampling of $\gamma$, $T_2$ that of $eta$.

Deviations from the paper: 3 rather than 20 datasets per setting; ESS is computed with ArviZ rather than the R package coda; only LIT-MH-1 is run (the paper additionally compares RW-MH and locally balanced variants — its Table 2 shows RW-MH acceptance rates of order $10^{-3}$ and ESS per second roughly an order of magnitude smaller).

In [ ]:
import time

import numpy as np
import pandas as pd
from numpy.linalg import cholesky
from scipy.linalg import solve_triangular

from BVS.DF_MCMC import DF_MCMC
from BVS.ESS import ess
from BVS.proposal import make_add, make_delete, weight

In [ ]:
def simulate_blocks(n, p, d, sigma_beta, seed):
    # paper sec. 5.3: rows iid MN(0, block-diag Sigma_{d,p}), blocks with
    # (Sigma_d)_jk = exp(-|j-k|/3); gamma* uniform of size 100,
    # beta*_{gamma*} ~ MN(0, sigma_beta^2 I)
    rng = np.random.default_rng(seed)
    j = np.arange(d)
    C = cholesky(np.exp(-np.abs(j[:, None] - j[None, :]) / 3))
    X = np.empty((n, p))
    for b0 in range(0, p, d):
        X[:, b0:b0 + d] = rng.standard_normal((n, d)) @ C.T
    gamma_star = rng.choice(p, size=100, replace=False)
    beta_star = np.zeros(p)
    beta_star[gamma_star] = sigma_beta * rng.standard_normal(100)
    Y = X @ beta_star[:, None] + rng.standard_normal((n, 1))
    return X, Y, beta_star

In [ ]:
def forward_backward(XTX, XTY, YTY, g, kappa, N, P):
    # greedy forward-backward stepwise initialization: keep adding (deleting)
    # the best single covariate while it increases the posterior
    gamma = np.zeros(P, dtype=bool)
    L, LZ, S = np.zeros((0, 0)), np.zeros(0), (1 + 1 / g) * YTY
    for _ in range(3):
        while True:
            if gamma.sum() < P:
                w_add, _ = weight(L, gamma, S, g, kappa, N, XTY, YTY, XTX, 0, p0=P)
                j = int(np.argmax(w_add))
                if w_add[j] > 1:
                    gamma, L, LZ, S = make_add(L, S, gamma, j, XTX, XTY, YTY, g)
                    continue
            if gamma.any():
                w_del, _ = weight(L, gamma, S, g, kappa, N, XTY, YTY, XTX, 1, p0=P)
                k = int(np.argmax(w_del))
                if w_del[k] > 1:
                    gamma, L, LZ, S = make_delete(L, S, gamma, k, XTX, XTY, YTY, g)
                    continue
            break
    return gamma

In [ ]:
def lit_run(X, Y, beta_star, gamma0, M, seed):
    """LIT-MH-1 via DF_MCMC (initialized at the stepwise model, beta draws
    sampled with Sample_beta=True); T1/T2 computed from the stored traces."""
    N, P = X.shape
    g = P ** (2 * KAPPA1) - 1
    q2 = g / (1 + g)
    XTX = X.T @ X
    XTYr = (X.T @ Y).ravel()

    t0 = time.perf_counter()
    beta_s, gamma_tr, sigma2_s = DF_MCMC(Y, X, KAPPA1, KAPPA2, P, 1, M=M,
                                         burn_in=0, seed=seed,
                                         Sample_beta=True, gamma_init=gamma0)
    wall = time.perf_counter() - t0

    T1 = np.empty(M)
    T2 = np.empty(M)
    cache = {}
    for t in range(M):
        key = tuple(np.flatnonzero(gamma_tr[:, t]).tolist())
        if key not in cache:
            pos = np.array(key)
            if pos.size:
                Lc = cholesky(XTX[np.ix_(pos, pos)])
                bh = q2 * solve_triangular(Lc.T, solve_triangular(Lc, XTYr[pos]))
                A = XTX[np.ix_(pos, pos)]
            else:
                pos, bh, A = pos, np.zeros(0), np.zeros((0, 0))
            cache[key] = (pos, bh, A)
        pos, bh, A = cache[key]
        full = np.zeros(P)
        full[pos] = bh
        T1[t] = ((full - beta_star) ** 2).sum()
        T2[t] = float(beta_s[pos, t] @ A @ beta_s[pos, t])

    # acceptance approximated by the fraction of iterations with a model change
    acc = float(np.mean(np.any(gamma_tr[:, 1:] != gamma_tr[:, :-1], axis=0)))
    return dict(T1=T1, T2=T2, sizes=gamma_tr.sum(axis=0), models=set(cache),
                acc=acc, time=wall)

In [ ]:
def local_modes(models, XTX, XTY, YTY, g, kappa, N, P):
    # gamma is a single-flip local mode if no add/delete neighbour has larger
    # posterior, i.e. every raw posterior ratio B(gamma, gamma') <= 1
    count = 0
    for key in models:
        gamma = np.zeros(P, dtype=bool)
        gamma[list(key)] = True
        pos = np.array(key)
        if pos.size:
            Lc = cholesky(XTX[np.ix_(pos, pos)])
            LZc = solve_triangular(Lc, XTY[gamma], lower=True)
            Sc = (1 + 1 / g) * YTY - (LZc ** 2).sum()
        else:
            Lc, Sc = np.zeros((0, 0)), (1 + 1 / g) * YTY
        w_add, _ = weight(Lc, gamma, Sc, g, kappa, N, XTY, YTY, XTX, 0, p0=P)
        w_del, _ = weight(Lc, gamma, Sc, g, kappa, N, XTY, YTY, XTX, 1, p0=P)
        if w_add.max() <= 1 and w_del.max() <= 1:
            count += 1
    return count

In [ ]:
SIGMA_BETAS = [0.1, 0.2, 0.3, 0.4, 0.5]   # paper Table 2 sweep
N_DATASETS = 3                            # paper: 20
M_LIT = 2000                              # paper: 2000
N, P, D = 1000, 5000, 20
KAPPA1, KAPPA2 = 0.5, 1.0                 # paper: kappa_1 = 1/2, kappa_0 = 1

rows = []
for sigma_beta in SIGMA_BETAS:
    vals = dict(time=[], size=[], modes=[], acc=[], t1=[], t2=[])
    for r in range(N_DATASETS):
        X, Y, beta_star = simulate_blocks(N, P, D, sigma_beta, seed=100 * r + 7)
        XTX = X.T @ X
        XTY = X.T @ Y
        YTY = (Y.T @ Y).item()
        g = P ** (2 * KAPPA1) - 1
        kappa = KAPPA1 + KAPPA2

        gamma0 = forward_backward(XTX, XTY, YTY, g, kappa, N, P)
        print(f"sigma_beta={sigma_beta}, dataset {r}: "
              f"forward-backward init size {gamma0.sum()}")

        res = lit_run(X, Y, beta_star, gamma0, M_LIT, seed=3000 * r + 11)
        vals["time"].append(res["time"])
        vals["size"].append(float(np.mean(res["sizes"])))
        vals["modes"].append(
            local_modes(res["models"], XTX, XTY, YTY, g, kappa, N, P))
        vals["acc"].append(res["acc"])
        vals["t1"].append(ess(res["T1"]) / res["time"])
        vals["t2"].append(ess(res["T2"]) / res["time"])

    rows.append(dict(
        sigma_beta=sigma_beta,
        time=float(np.mean(vals["time"])),
        model_size=float(np.mean(vals["size"])),
        local_modes=float(np.mean(vals["modes"])),
        acc_rate=float(np.mean(vals["acc"])),
        **{"ESS(T1)/time": float(np.mean(vals["t1"])),
           "ESS(T2)/time": float(np.mean(vals["t2"]))},
    ))
    print(pd.DataFrame(rows).tail(1).round(3).to_string(index=False))

In [ ]:
df = pd.DataFrame(rows)
df

## Notes

Expected agreement with the LIT-MH-1 column of the paper's Table 2:

- the acceptance rate stays around $0.5$–$0.6$ across all $\sigma_eta$ (for contrast, the paper's RW-MH column drops from 0.012 to 0.002);
- the number of unique local modes sampled peaks around $\sigma_eta = 0.2$, where the posterior is most multi-modal (paper: 6.2; RW-MH sampled only 2.6);
- the mean model size grows from about 6 to about 72 as $\sigma_eta$ increases (paper: 6.1 → 71.6);
- per second of wall time, the effective sample sizes of both $T_1$ (sampling $\gamma$) and $T_2$ (sampling $eta$) are roughly an order of magnitude above the paper's RW-MH column — the dimension-free mixing claim of Theorem 1 in action.